# Phase 0 — Lesson 12: Debugging and Profiling
> The worst AI bugs don't crash. They train silently on garbage and report a beautiful loss curve.

**Source:** ai-engineering-from-scratch -> Phase 0 (Setup & Tooling), Lesson 12
**Type:** Build | **Language:** Python | **Time:** ~60 minutes
**Prerequisites:** Lesson 1 (Dev Environment), basic PyTorch familiarity

This lesson explains *how* AI bugs are different from regular code bugs, *what* tools catch them, and *when* to use each debugging level. Run the code cells to see visualizations.


# The Problem: AI Code Fails Silently

---


A web app crashes with a stack trace. You know exactly what went wrong.

AI code fails differently. A misconfigured training loop runs for 8 hours, burns $200 in GPU time, and produces a model that predicts the mean of every input. **The code never errored.** The bug was a tensor on the wrong device, a forgotten `.detach()`, or labels leaking into features.

You need debugging tools that catch these silent failures before they waste your time and compute.


# The Concept: Three Levels of AI Debugging

---


AI debugging operates at three levels. Most people jump straight to level 3 (staring at TensorBoard). But **80% of AI bugs live at levels 1 and 2**.

| Level | What | Tools |
|-------|------|-------|
| 1. Standard Python | Breakpoints, logging, profiling, memory | `breakpoint()`, `logging`, `cProfile`, `tracemalloc` |
| 2. Tensor Operations | Shapes, dtypes, devices, NaN/Inf values | `debug_print`, tensor inspection, hooks |
| 3. Training Dynamics | Loss curves, gradient norms, activations | TensorBoard, weight histograms |


# Visual: Three Levels — Where Bugs Actually Live

---


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(9, 4))
ax.axis('off')

levels = [
    ('Level 1: Standard Python', 'Breakpoints, logging, profiling\n80% of bugs live here', '#27ae60'),
    ('Level 2: Tensor Operations', 'Shapes, dtypes, devices, NaN/Inf', '#f39c12'),
    ('Level 3: Training Dynamics', 'Loss curves, gradients, activations', '#e74c3c'),
]

y = 0.85
for label, desc, color in levels:
    rect = patches.Rectangle((0.1, y-0.09), 0.8, 0.16, facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(0.15, y, label, ha='left', va='center', color='white', fontweight='bold', fontsize=11)
    ax.text(0.85, y, desc, ha='right', va='center', fontsize=9, style='italic')
    y -= 0.2

for i in range(2):
    ax.annotate('', xy=(0.5, y+0.02), xytext=(0.5, y+0.06),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    y -= 0.2

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Three Debugging Levels — Start at Level 1", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **Level 1 and 2 bugs** are caught by printing tensor shapes, checking devices, and looking for NaN values. **Level 3** tells you something is wrong but not what.


# Part 1: Print Debugging (Yes, It Works)

---


Print debugging gets dismissed. It shouldn't. For tensor code, a targeted print statement beats stepping through a debugger because you need to see shapes, dtypes, and value ranges all at once.

```python
def debug_print(name, tensor):
    print(f"{name}: shape={tensor.shape}, dtype={tensor.dtype}, "
          f"device={tensor.device}, "
          f"min={tensor.min().item():.4f}, max={tensor.max().item():.4f}, "
          f"mean={tensor.mean().item():.4f}, "
          f"has_nan={tensor.isnan().any().item()}")

# Call this after every suspicious operation
debug_print("outputs", outputs)
debug_print("gradients", loss.grad)
```

When the bug is found, remove the prints. Simple.


# Part 2: Python Debugger (pdb and breakpoint)

---


The built-in debugger is underrated for AI work. Drop `breakpoint()` into your training loop and inspect tensors interactively.

```python
def training_step(model, batch, criterion, optimizer):
    inputs, labels = batch
    outputs = model(inputs)
    loss = criterion(outputs, labels)

    if loss.item() > 100 or torch.isnan(loss):
        breakpoint()

    loss.backward()
    optimizer.step()
```

When the debugger drops you in, useful commands:
- `p outputs.shape` to check shapes
- `p loss.item()` to see the loss value
- `p torch.isnan(outputs).sum()` to count NaNs
- `p model.fc1.weight.grad` to check gradients
- `c` to continue, `q` to quit

This is **conditional debugging**. You only stop when something looks wrong. For a 10,000-step training run, that matters.


# Part 3: Python Logging

---


Replace print statements with logging when your debugging goes beyond a quick check.

```python
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("training.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

logger.info("Starting training: lr=%.4f, batch_size=%d", lr, batch_size)
logger.warning("Loss spike detected: %.4f at step %d", loss.item(), step)
logger.error("NaN loss at step %d, stopping", step)
```

Logging gives you timestamps, severity levels, and file output. When a training run fails at 3 AM, you want a log file, not terminal output that scrolled off screen.


# Part 4: Timing Code Sections

---


Knowing where time goes is the first step to optimization.

```python
import time

class Timer:
    def __init__(self, name=""):
        self.name = name

    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, *args):
        elapsed = time.perf_counter() - self.start
        print(f"[{self.name}] {elapsed:.4f}s")

with Timer("data loading"):
    batch = next(dataloader_iter)

with Timer("forward pass"):
    outputs = model(batch)

with Timer("backward pass"):
    loss.backward()
```

**Common finding:** data loading takes 60% of training time. The fix is `num_workers > 0` in your DataLoader, not a faster GPU.


# Visual: Typical Training Loop Time Breakdown

---


In [ ]:
import matplotlib.pyplot as plt

components = ['Data Loading', 'Forward Pass', 'Backward Pass', 'Optimizer Step', 'Logging']
times = [60, 20, 15, 4, 1]  # percent
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#95a5a6']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(components, times, color=colors, alpha=0.8)
ax.set_ylabel('% of Training Time')
ax.set_title('Where Training Time Goes (num_workers=0)', fontsize=12, fontweight='bold')
ax.set_ylim(0, 70)
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1, f'{val}%', ha='center', fontweight='bold')

ax.text(0.5, 0.02, 'Fix: set num_workers > 0 in DataLoader', transform=ax.transAxes,
        ha='center', fontsize=10, fontweight='bold', color='#27ae60')

plt.tight_layout()
plt.show()


> **Data loading dominates.** Profile before you optimize — don't guess.


# Part 5: cProfile and line_profiler

---


When you need more than manual timers:

```bash
python -m cProfile -s cumtime train.py
```

This shows every function call sorted by cumulative time. For line-by-line profiling:

```bash
pip install line_profiler
```

```python
@profile
def train_step(model, data, target):
    output = model(data)
    loss = F.cross_entropy(output, target)
    loss.backward()
    return loss

# Run with: kernprof -l -v train.py
```


# Part 6: Memory Profiling

---


### CPU Memory with tracemalloc

```python
import tracemalloc

tracemalloc.start()

# your code here
model = build_model()
data = load_dataset()

snapshot = tracemalloc.take_snapshot()
top_stats = snapshot.statistics("lineno")
for stat in top_stats[:10]:
    print(stat)
```

### CPU Memory with memory_profiler

```bash
pip install memory_profiler
```

```python
from memory_profiler import profile

@profile
def load_data():
    raw = read_csv("data.csv")       # watch memory jump here
    processed = preprocess(raw)       # and here
    return processed
```

Run with `python -m memory_profiler your_script.py` to see line-by-line memory usage.

### GPU Memory with PyTorch

```python
import torch

if torch.cuda.is_available():
    print(torch.cuda.memory_summary())
    
    print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"Cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
```

**When you hit OOM (Out of Memory):**
1. Reduce batch size (first thing to try, always)
2. Use `torch.cuda.empty_cache()` to free cached memory
3. Use `del tensor` followed by `torch.cuda.empty_cache()` for large intermediates
4. Use mixed precision (`torch.cuda.amp`) to halve memory usage
5. Use gradient checkpointing for very deep models


# Part 7: Common AI Bugs and How to Catch Them

---


### Shape Mismatch

The most frequent bug. A tensor has shape `[batch, features]` when the model expects  `[batch, channels, height, width]`.

```python
def check_shapes(model, sample_input):
    print(f"Input: {sample_input.shape}")
    hooks = []

    def make_hook(name):
        def hook(module, inp, out):
            in_shape = inp[0].shape if isinstance(inp, tuple) else inp.shape
            out_shape = out.shape if hasattr(out, "shape") else type(out)
            print(f"  {name}: {in_shape} -> {out_shape}")
        return hook

    for name, module in model.named_modules():
        hooks.append(module.register_forward_hook(make_hook(name)))

    with torch.no_grad():
        model(sample_input)

    for h in hooks:
        h.remove()
```

Run this once with a sample batch. It maps every shape transformation in your model.

### NaN Loss

NaN loss means something exploded. Common causes:
- Learning rate too high
- Division by zero in custom loss
- Log of zero or negative number
- Exploding gradients in RNNs

```python
def detect_nan(model, loss, step):
    if torch.isnan(loss):
        print(f"NaN loss at step {step}")
        for name, param in model.named_parameters():
            if param.grad is not None:
                if torch.isnan(param.grad).any():
                    print(f"  NaN gradient in {name}")
                if torch.isinf(param.grad).any():
                    print(f"  Inf gradient in {name}")
        return True
    return False
```

### Data Leakage

Your model gets 99% accuracy on the test set. Sounds great. It's a bug.

```python
def check_data_leakage(train_set, test_set, id_column="id"):
    train_ids = set(train_set[id_column].tolist())
    test_ids = set(test_set[id_column].tolist())
    overlap = train_ids & test_ids
    if overlap:
        print(f"DATA LEAKAGE: {len(overlap)} samples in both train and test")
        return True
    return False
```

Also check for **temporal leakage**: using future data to predict the past. Sort by timestamp before splitting.

### Wrong Device

Tensors on different devices (CPU vs GPU) cause runtime errors. But sometimes a tensor silently stays on CPU while everything else is on GPU, and training just runs slowly.

```python
def check_devices(model, *tensors):
    model_device = next(model.parameters()).device
    print(f"Model device: {model_device}")
    for i, t in enumerate(tensors):
        if t.device != model_device:
            print(f"  WARNING: tensor {i} on {t.device}, model on {model_device}")
```


# Visual: Most Common AI Bugs by Frequency

---


In [ ]:
import matplotlib.pyplot as plt

bugs = ['Shape mismatch', 'NaN loss', 'Wrong device', 'Data leakage', 'Gradient explosion']
frequency = [40, 25, 15, 12, 8]  # relative frequency

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#e74c3c', '#e67e22', '#f39c12', '#9b59b6', '#34495e']
bars = ax.barh(bugs, frequency, color=colors, alpha=0.8)
ax.set_xlabel('Relative Frequency (%)')
ax.set_title('Most Common AI Bugs', fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

for bar, val in zip(bars, frequency):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2, f'{val}%', va='center', fontweight='bold')

ax.set_xlim(0, 50)
plt.tight_layout()
plt.show()


> **Shape mismatch is #1.** Always check tensor shapes with `debug_print` or forward hooks before training.


# Part 8: TensorBoard Basics

---


TensorBoard shows you what's happening inside training over time.

```bash
pip install tensorboard
```

```python
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter("runs/experiment_1")

for step in range(num_steps):
    loss = train_step(model, batch)

    writer.add_scalar("loss/train", loss.item(), step)
    writer.add_scalar("lr", optimizer.param_groups[0]["lr"], step)

    if step % 100 == 0:
        for name, param in model.named_parameters():
            writer.add_histogram(f"weights/{name}", param, step)
            if param.grad is not None:
                writer.add_histogram(f"grads/{name}", param.grad, step)

writer.close()
```

Launch it:
```bash
tensorboard --logdir=runs
```

**What to look for:**
- **Loss not decreasing**: Learning rate too low, or model architecture issue
- **Loss oscillating wildly**: Learning rate too high
- **Loss goes to NaN**: Numerical instability
- **Train loss decreasing, val loss increasing**: Overfitting
- **Weight histograms collapsing to zero**: Vanishing gradients
- **Gradient histograms exploding**: Need gradient clipping


# Part 9: VS Code Debugger

---


For interactive debugging, configure VS Code with a `launch.json`:

```json
{
    "version": "0.2.0",
    "configurations": [
        {
            "name": "Debug Training",
            "type": "debugpy",
            "request": "launch",
            "program": "${file}",
            "console": "integratedTerminal",
            "justMyCode": false
        }
    ]
}
```

Set breakpoints by clicking the gutter. Use the Variables pane to inspect tensor properties. The Debug Console lets you run arbitrary Python expressions mid-execution.

Useful for stepping through data preprocessing pipelines where you want to see each transformation.


# Use It: The Debugging Workflow

---


Here's the debugging workflow that catches most AI bugs:

1. **Before training**: Run `check_shapes` with a sample batch. Verify input and output dimensions match expectations.
2. **First 10 steps**: Use `debug_print` on loss, outputs, and gradients. Confirm nothing is NaN and values are in reasonable ranges.
3. **During training**: Log loss, learning rate, and gradient norms. Use TensorBoard for visualization.
4. **When something breaks**: Drop `breakpoint()` at the failure point. Inspect tensors interactively.
5. **For performance**: Time your data loading vs forward vs backward pass. Profile memory if you're near OOM.


# Quiz Review

---


Try answering these before reading the explanations:

**Q1:** What makes debugging AI/ML code fundamentally different from debugging a typical web application?
> **Explanation:** AI bugs often don't crash — they silently produce incorrect results with no error messages. A misconfigured training loop might run for hours without errors while the model learns nothing useful.

**Q2:** What does a profiler measure?
> **Explanation:** How much time and memory each part of the code consumes. Profilers help you find bottlenecks.

**Q3:** Your model achieves 99% accuracy on the test set. What AI-specific bug should you suspect first?
> **Explanation:** Data leakage — test samples may have leaked into the training set, or features contain the target label.

**Q4:** What is the most common finding when profiling a training loop's time breakdown?
> **Explanation:** Data loading takes more time than the forward and backward passes combined. Fix: set `num_workers > 0` in the DataLoader.

**Q5:** You see NaN loss at step 500. Which approach will help you find the root cause?
> **Explanation:** Use `detect_nan` to check for NaN/Inf in gradients and add `breakpoint()` at the failure point. Common causes: learning rate too high, log(0), or division by zero.


# Practice Exercises

---


Do these in order:

**Exercise 1:** Run `debug_tools.py` and read through each section's output. Modify the dummy model to introduce a NaN (hint: divide by zero in the forward pass) and watch the detector catch it.

**Exercise 2:** Profile a training loop with `cProfile` and identify the slowest function.

**Exercise 3:** Use `tracemalloc` to find which line in your data loading pipeline allocates the most memory.

**Exercise 4:** Set up TensorBoard for a simple training run and identify whether the model is overfitting.

**Exercise 5:** Use `breakpoint()` inside a training loop. Practice inspecting tensor shapes, devices, and gradient values.

```bash
python phases/00-setup-and-tooling/12-debugging-and-profiling/code/debug_tools.py
```


# Key Terms

---


| Term | What people say | What it actually means |
|------|----------------|----------------------|
| breakpoint() | "Pause button" | Drops into an interactive Python debugger at that line |
| Profile | "Stopwatch for code" | Measures time and memory consumed by each function |
| tracemalloc | "Memory tracker" | Tracks Python memory allocations line by line |
| TensorBoard | "Training dashboard" | Visualizes loss curves, weight histograms, and gradient distributions |
| OOM | "Out of memory" | GPU runs out of VRAM. Fix: reduce batch size, use mixed precision |
| Data leakage | "Cheating" | Test data appears in training, inflating accuracy artificially |


# Next Steps

---


You've completed Phase 0: Setup & Tooling. You're ready for Phase 1: Math Foundations.

First lesson:
```bash
python3 phases/01-math-foundations/01-linear-algebra-intuition/code/vectors.py
```

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
